# Fine-tuning a masked language model

## Load Dataset:

In [1]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

## Load Tokenizer:

In [2]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

## Load Model:

In [12]:
from transformers import AutoModelForMaskedLM
import torch

ckpt = "distilbert-base-uncased"

base_model = AutoModelForMaskedLM.from_pretrained(ckpt)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [13]:
from transformers import AutoModelForMaskedLM
import torch

finetuned_ckpt = "tankgauravgt/distilbert-uncased-imdb-finetuned"

finetuned_model = AutoModelForMaskedLM.from_pretrained(finetuned_ckpt)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [22]:
import torch

text = "spider[MASK]."

inputs = tokenizer(text, return_tensors="pt")
token_logits = base_model(**inputs).logits

# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> spiderbites.'
'>>> spiderweb.'
'>>> spiderbite.'
'>>> spiderspecies.'
'>>> spidermonkeys.'


In [23]:
import torch

text = "spider[MASK]."

inputs = tokenizer(text, return_tensors="pt")
token_logits = finetuned_model(**inputs).logits

# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> spiderbites.'
'>>> spiderweb.'
'>>> spiderbite.'
'>>> spiderspecies.'
'>>> spidermonkey.'


## Tokenize Dataset:

In [4]:
tokenized_datasets = raw_datasets.map(
    function=lambda x: tokenizer(x['text'], truncation=True, max_length=512), 
    batched=True, 
    remove_columns=["text", "label"]
)
tokenized_datasets

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

## Data Collator:

In [ ]:
import random
import torch
from transformers import DataCollatorForLanguageModeling


class WholeWordMaskingDataCollator:
    """Mask 15% of WHOLE words (all sub-tokens of a selected word get masked).

    Word-piece tokenization marks sub-word continuations with '##' prefix.
    We group tokens by word boundary, then randomly select 15% of words
    and mask ALL their sub-tokens.
    """

    def __init__(self, tokenizer, mlm_probability=0.15):
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability
        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id
        self.special_tokens = {
            tokenizer.cls_token_id, tokenizer.sep_token_id,
            tokenizer.pad_token_id, tokenizer.mask_token_id,
        }

    def _get_word_groups(self, input_ids):
        """Group token indices by whole word using '##' prefix detection."""
        groups = []
        current_group = []
        for idx, token_id in enumerate(input_ids):
            if token_id in self.special_tokens:
                if current_group:
                    groups.append(current_group)
                    current_group = []
                continue
            token_str = self.tokenizer.convert_ids_to_tokens(token_id)
            if token_str.startswith("##"):
                current_group.append(idx)
            else:
                if current_group:
                    groups.append(current_group)
                current_group = [idx]
        if current_group:
            groups.append(current_group)
        return groups

    def __call__(self, batch):
        input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
        attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
        labels = input_ids.clone()

        # Pad to same length
        input_ids = input_ids.masked_fill(attention_mask == 0, self.pad_token_id)

        for i in range(input_ids.shape[0]):
            word_groups = self._get_word_groups(input_ids[i].tolist())

            # Sample 15% of whole words to mask
            num_to_mask = max(1, int(len(word_groups) * self.mlm_probability))
            words_to_mask = random.sample(word_groups, min(num_to_mask, len(word_groups)))

            # Mask all sub-tokens of selected words
            for group in words_to_mask:
                for idx in group:
                    labels[i, idx] = input_ids[i, idx]       # keep original as label
                    input_ids[i, idx] = self.mask_token_id    # replace with [MASK]

            # Ignore padding in labels
            labels[i, attention_mask[i] == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


data_collator = WholeWordMaskingDataCollator(tokenizer=tokenizer, mlm_probability=0.15)

## Domain Adapt Model:

In [6]:
from transformers import TrainingArguments
from transformers import Trainer

In [7]:
args = TrainingArguments(
    output_dir="distilbert-mlm-imdb",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=256,
    bf16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["unsupervised"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [8]:
import math

perplexity = math.exp(trainer.evaluate()["eval_loss"])
perplexity

Training Loss,Validation Loss,Epoch
No log,2.837168,0


17.067353751596368

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.327325
2,No log,2.322924
3,2.415480,2.315938


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=588, training_loss=2.412182788459622, metrics={'train_runtime': 322.8793, 'train_samples_per_second': 464.57, 'train_steps_per_second': 1.821, 'total_flos': 1.98841734144e+16, 'train_loss': 2.412182788459622, 'epoch': 3.0})

In [10]:
import math

perplexity = math.exp(trainer.evaluate()["eval_loss"])
perplexity

Training Loss,Validation Loss,Epoch
2.415480,2.320278,3


10.178502817216474

In [12]:
from huggingface_hub import create_repo

# 1. Define your model ID
repo_id = "tankgauravgt/distilbert-cased-imdb-finetuned"
trainer.hub_model_id = repo_id

# 2. Create the repository on the Hub (safe if it already exists)
create_repo(repo_id=repo_id, exist_ok=True)

# 3. Push your model and logs to the Hub
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/tankgauravgt/distilbert-cased-imdb-finetuned/commit/14d8883ae055ca8bf64f57410a31e434d710c94f', commit_message='End of training', commit_description='', oid='14d8883ae055ca8bf64f57410a31e434d710c94f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tankgauravgt/distilbert-cased-imdb-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='tankgauravgt/distilbert-cased-imdb-finetuned'), pr_revision=None, pr_num=None)